# 01d — Reasoning-effort runs (data generation only)

Run GPT-5 no-desc on `tuning_sample.csv` (the tuning set) at three values of
`reasoning_effort`: `low`, `medium` (default), and `high`.  Three API keys are
used so the three runs go in parallel, avoiding per-key rate limits.

**Outputs (saved to `data/results/llm/`):**
- `01d_predictions.csv` — one row per loan × variant: `actual`, `llm_pred`, `prob_fully_paid`, `reasoning_effort`, `llm_reasoning`
- `01d_metrics.csv` — accuracy / precision / recall / F1 / AUC per variant (default-threshold view)
- Per-call rows are also auto-appended to `llm_calls.csv`.

This notebook **only generates data**.  Threshold tuning, the medium-vs-high
decision, and the held-out validation all happen in `04b_Final_Benchmark.ipynb`.

In [1]:
# llm_utils.py and llm_pricing.py live one directory up — make them importable.
import sys; sys.path.insert(0, "..")

import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    compare_results, evaluate_predictions, RESULTS_DIR,
)

In [2]:
# Load OpenAI API keys from .env — one key per effort level, no sharing.
load_dotenv("../.env", override=True)

_key1 = os.environ.get("OPENAI_API_KEY")
_key2 = os.environ.get("OPENAI_API_KEY_2") or _key1
_key3 = os.environ.get("OPENAI_API_KEY_3") or _key1

API_KEYS = {
    "low":    _key1,
    "medium": _key2,
    "high":   _key3,   # each effort on its own key — no sharing
}
missing = [k for k, v in API_KEYS.items() if not v]
assert not missing, f"Missing API keys for: {missing}. Add them to ../.env."
n_unique = len(set(API_KEYS.values()))
print(f"API keys loaded ({n_unique} unique key(s), one per effort level).")

API keys loaded (3 unique key(s), one per effort level).


In [3]:
# Tuning sample = the 100-loan LLM eval sample (built by 02_Preprocessing).
# Threshold tuning will use these labels in the next notebook.
tuning_sample = load_llm_sample()
y_true = tuning_sample['loan_status'].values
print(f"Tuning sample: {len(tuning_sample)} loans")
print(f"  Charged Off: {(y_true == 0).sum()}")
print(f"  Fully Paid:  {(y_true == 1).sum()}")

# XGBoost predictions on the same sample for context (printed only).
xgb_probs, xgb_preds = run_ml_on_sample(tuning_sample)
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(),
                                   label="XGBoost (tuning sample)",
                                   probabilities=xgb_probs.tolist())

Tuning sample: 100 loans
  Charged Off: 15
  Fully Paid:  85

XGBoost (tuning sample) Results (100 samples)
Accuracy: 65.0%
AUC:      0.671  (over 100 rows with logprobs)

Classification Report:
              precision    recall  f1-score   support

 Charged Off       0.21      0.47      0.29        15
  Fully Paid       0.88      0.68      0.77        85

    accuracy                           0.65       100
   macro avg       0.54      0.57      0.53       100
weighted avg       0.78      0.65      0.70       100

Confusion Matrix:
[[ 7  8]
 [27 58]]


In [4]:
# Run three GPT-5.4 variants concurrently, each with its own API key.
MODEL = "gpt-5.4"
EFFORTS = ["low", "medium", "high"]

def run_one(effort):
    return run_llm_experiment(
        tuning_sample,
        api_provider="openai",
        model_name=MODEL,
        api_key=API_KEYS[effort],
        label=f"GPT-5.4 reasoning={effort}",
        include_desc=False,
        with_logprobs=True,
        reasoning_effort=effort,
    )

results = {}
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(run_one, e): e for e in EFFORTS}
    for fut in futures:
        e = futures[fut]
        results[e] = fut.result()

print("\nAll three runs complete.")

/Users/alemz/Projects/Github/Sabadell_Capstone/notebooks/llm_models/01_model_selection/../llm_utils.py:1020: UserWarning: OpenAI model gpt-5.4 does not support logprobs under current settings. Falling back to calling without logprobs.
  return call_llm(


[GPT-5.4 reasoning=low | no_desc] First call OK (pred=1, prob=None). Starting full run...
[GPT-5.4 reasoning=high | no_desc] First call OK (pred=1, prob=None). Starting full run...
[GPT-5.4 reasoning=medium | no_desc] First call OK (pred=1, prob=None). Starting full run...
[GPT-5.4 reasoning=low | no_desc] 10/100 done (28s elapsed, ~256s remaining)
[GPT-5.4 reasoning=medium | no_desc] 10/100 done (47s elapsed, ~424s remaining)
[GPT-5.4 reasoning=low | no_desc] 20/100 done (54s elapsed, ~218s remaining)
[GPT-5.4 reasoning=high | no_desc] 10/100 done (61s elapsed, ~552s remaining)
[GPT-5.4 reasoning=low | no_desc] 30/100 done (86s elapsed, ~200s remaining)
[GPT-5.4 reasoning=medium | no_desc] 20/100 done (95s elapsed, ~380s remaining)
[GPT-5.4 reasoning=low | no_desc] 40/100 done (115s elapsed, ~172s remaining)
[GPT-5.4 reasoning=low | no_desc] 50/100 done (144s elapsed, ~144s remaining)
[GPT-5.4 reasoning=high | no_desc] 20/100 done (142s elapsed, ~567s remaining)
[GPT-5.4 reasoning=med

In [ ]:
# Build a single consolidated predictions CSV (one row per loan × variant).
# Per-loan tokens/cost are embedded so cost is derivable from this file alone.
rows = []
for effort, res in results.items():
    ti, to, cu = res.get('input_tokens'), res.get('output_tokens'), res.get('cost_usd')
    for i in range(len(tuning_sample)):
        rows.append({
            "row_index":         i,
            "reasoning_effort":  effort,
            "actual":            int(y_true[i]),
            "llm_pred":          res['predictions'][i],
            "prob_fully_paid":   res['probabilities'][i],
            "llm_reasoning":     res['reasonings'][i],
            "xgb_pred":          int(xgb_preds[i]),
            "xgb_prob":          float(xgb_probs[i]),
            "input_tokens":      ti[i] if ti else None,
            "output_tokens":     to[i] if to else None,
            "cost_usd":          cu[i] if cu else None,
        })

predictions_df = pd.DataFrame(rows)
out_path = f"{RESULTS_DIR}/01d_predictions.csv"
predictions_df.to_csv(out_path, index=False)
print(f"Saved {len(predictions_df)} rows to {out_path}")

In [6]:
# Summary metrics per variant at the LLM's default (hard 0/1) prediction.
# These are the BEFORE-tuning numbers; tuned-threshold metrics are in 04b_Final_Benchmark.ipynb.
metrics_rows = []
for effort, res in results.items():
    m = res['metrics'].copy()
    m['reasoning_effort'] = effort
    m['variant'] = f"GPT-5.4 reasoning={effort}"
    metrics_rows.append(m)

metrics_rows.append({**xgb_metrics, 'reasoning_effort': '-', 'variant': 'XGBoost (tuning)'})

metrics_df = pd.DataFrame(metrics_rows)
cols = ['variant', 'reasoning_effort', 'accuracy', 'auc',
        'precision_charged_off', 'recall_charged_off', 'f1_charged_off']
metrics_df = metrics_df[[c for c in cols if c in metrics_df.columns]]

out_path = f"{RESULTS_DIR}/01d_metrics.csv"
metrics_df.to_csv(out_path, index=False)
print(metrics_df.to_string(index=False))
print(f"\nSaved metrics to {out_path}")

               variant reasoning_effort  accuracy      auc  precision_charged_off  recall_charged_off  f1_charged_off
   GPT-5 reasoning=low              low      0.74      NaN               0.280000            0.466667        0.350000
GPT-5 reasoning=medium           medium      0.76      NaN               0.263158            0.333333        0.294118
  GPT-5 reasoning=high             high      0.80      NaN               0.352941            0.400000        0.375000
      XGBoost (tuning)                -      0.65 0.670588               0.205882            0.466667        0.285714

Saved metrics to /Users/alemz/Projects/Github/Sabadell_Capstone/data/results/llm/01d_metrics.csv
